In [4]:
def trigger_working_set_manager(state):
    print(f"ALERT!: Thrashing detected with severity {state:.3f}! Triggering memory control module Immediately")
import time
import joblib
import psutil
import pandas as pd
from collections import deque
model = joblib.load("thrashing_detection_model.pkl")
scaler = joblib.load("thrashing_detection_scaler.pkl")
old_disk = psutil.disk_io_counters()
recent_thrashing_probs=deque(maxlen=5) 
curr_state="NORMAL"
recovers_in=5
recovery_counter=0
safe=0.3
try:
    while True:
        cpu = psutil.cpu_percent(interval=1)
        mem = psutil.virtual_memory()
        swap = psutil.swap_memory()
        new_disk = psutil.disk_io_counters()

        if old_disk and new_disk:
            disk_reads_per_sec = new_disk.read_count - old_disk.read_count
            disk_writes_per_sec = new_disk.write_count - old_disk.write_count
        else:
            disk_reads_per_sec = 0
            disk_writes_per_sec = 0
        old_disk = new_disk
        features = pd.DataFrame([[cpu, mem.percent, swap.percent,
                                   disk_reads_per_sec, disk_writes_per_sec]],
                                 columns=['cpu_percent','ram_percent','swap_percent',
                                          'disk_read_count','disk_write_count'])

        scaled_features = scaler.transform(features)
        probabilities = model.predict_proba(scaled_features)
        print(probabilities)
        p_normal, p_warning, p_thrashing = probabilities[0]
        recent_thrashing_probs.append(p_thrashing)
        avg_thrashing = sum(recent_thrashing_probs) / len(recent_thrashing_probs)
        if curr_state in ("NORMAL", "WARNING"):
            if avg_thrashing >= 0.75:
                curr_state = "THRASHING"
                trigger_working_set_manager(avg_thrashing)
            elif p_warning >= p_normal and p_warning > p_thrashing:
                curr_state = "WARNING"
            else:
                curr_state = "NORMAL"

        elif curr_state == "THRASHING":
            if avg_thrashing < 0.75:
                curr_state = "RECOVERY"
                recovery_counter = 0
        elif curr_state == "RECOVERY":
            if avg_thrashing >= 0.75:
                curr_state = "THRASHING"
                trigger_working_set_manager(avg_thrashing)
                recovery_counter = 0
            elif avg_thrashing < safe:
                recovery_counter += 1
                if recovery_counter >= recovers_in:
                    curr_state = "NORMAL"
                    recovery_counter=0
            else:
                recovery_counter=0

        print(curr_state)
except KeyboardInterrupt:
    print("Running stopped by user")

[[1.16368307e-04 9.99883632e-01 2.68858637e-32]]
WARNING
[[2.62898007e-05 9.99973710e-01 4.05115996e-32]]
WARNING
[[2.11722461e-04 9.99788278e-01 2.38899154e-32]]
WARNING
[[2.31503469e-04 9.99768497e-01 2.31104689e-32]]
WARNING
[[2.09957745e-04 9.99790042e-01 2.32030365e-32]]
WARNING
[[1.98713766e-04 9.99801286e-01 2.42434709e-32]]
WARNING
[[2.88814771e-04 9.99711185e-01 2.20403876e-32]]
WARNING
[[2.64578201e-04 9.99735422e-01 2.23373360e-32]]
WARNING
[[1.88377086e-04 9.99811623e-01 2.45756205e-32]]
WARNING
[[1.92991634e-04 9.99807008e-01 2.43518050e-32]]
WARNING
[[7.86075309e-05 9.99921392e-01 9.04390963e-33]]
WARNING
[[2.09421806e-05 9.99979058e-01 1.35029716e-32]]
WARNING
[[5.28188246e-05 9.99947181e-01 1.05740255e-32]]
WARNING
[[7.22957042e-05 9.99927704e-01 9.87854895e-33]]
WARNING
[[8.15775029e-05 9.99918422e-01 9.83995756e-33]]
WARNING
[[3.95303413e-05 9.99960470e-01 9.75608144e-33]]
WARNING
[[8.41389308e-05 9.99915861e-01 9.84079627e-33]]
WARNING
[[9.01399212e-05 9.99909860e-01

In [7]:
model.coef_

array([[-0.75844566, -1.54399309, -1.49108716, -0.44405726, -0.08132077],
       [ 0.25080267,  0.05905057, -2.31692982,  0.55183188,  0.14771064],
       [ 0.50764299,  1.48494252,  3.80801698, -0.10777461, -0.06638987]])